# What Drives Altcoin Returns?

## Rolling crypto style analysis with Bitcoin, Ethereum, Dow Jones, and gold

**15-minute project presentation | Associate Data Scientist – Metyis**

### Executive answer

- **Crypto factors dominate:** Ethereum and Bitcoin provide the largest style exposures across the target coins.
- **Replication is incomplete:** the four-factor model leaves meaningful asset-specific variation unexplained.
- **Behavior changes through time:** rolling estimates reveal material changes in benchmark exposures and model fit.

> **Core message:** these altcoins share substantial crypto-market beta, but they are not interchangeable. Asset-specific risk remains important for portfolio construction and risk monitoring.

## 1. Business question and analytical framing — 1 minute

**Question:** Can the daily returns of six major altcoins be replicated by a transparent mix of broad market factors?

**Targets:** XRP, Litecoin, BNB, DOGE, Cardano, and Solana.  
**Factors:** Bitcoin, Ethereum, Dow Jones Industrial Average, and gold.

This matters because a portfolio manager or risk team needs to know whether an altcoin adds differentiated exposure or mostly repackages risks already present elsewhere. The output can support:

- factor-based risk attribution;
- benchmark and hedge selection;
- diversification monitoring;
- investigation of changing market regimes.

The exact balanced-sample dates are reported in the data audit below.

In [ ]:
# Standard-library imports keep paths portable across local and interview environments.
from pathlib import Path
import sys
import warnings

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import numpy as np
import pandas as pd
from IPython.display import display

# Resolve the repository root. The fallback supports launching Jupyter from the
# parent directory rather than from the project directory itself.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    candidate = PROJECT_ROOT.parent / 'crypto_style_analysis'
    if candidate.exists():
        PROJECT_ROOT = candidate

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.data_analysis import create_adf_result_data_frame
from src.preprocess_data import crypto_series_to_process
from src.download_data import ensure_yahoo_data_coverage
from src.style_analysis import rolling_style_analysis

warnings.filterwarnings('ignore')

# Matplotlib renamed its bundled Seaborn styles in version 3.6. Try the modern
# name first, then fall back to the name used by the project's Python 3.7 setup.
try:
    plt.style.use('seaborn-v0_8-whitegrid')
except OSError:
    plt.style.use('seaborn-whitegrid')

pd.set_option('display.max_columns', 20)

DATA_DIR = PROJECT_ROOT / 'data'
START_DATE = '2019-01-03'  # Extra history is needed to calculate the first return.
END_DATE = '2026-08-31'
WINDOW_SIZE = 300           # Trailing observations in each style estimate.
STEP_SIZE = 7               # Weekly estimation keeps the presentation run time short.

# Local CSV snapshots make the presentation reproducible and remove live-API risk.
TICKERS = [
    'BTC-USD', 'ETH-USD', 'XRP-USD', 'LTC-USD', 'BNB-USD',
    'DOGE-USD', 'ADA-USD', 'SOL-USD', '^DJI', 'GC=F'
]
BENCHMARKS = [
    'log_return_BTC-USD', 'log_return_ETH-USD',
    'log_return_^DJI', 'log_return_GC=F'
]
TARGETS = [
    'log_return_XRP-USD', 'log_return_LTC-USD', 'log_return_BNB-USD',
    'log_return_DOGE-USD', 'log_return_ADA-USD', 'log_return_SOL-USD'
]
ALL_RETURNS = BENCHMARKS + TARGETS

# `str.removeprefix` is unavailable in Python 3.7, so use a one-time replacement.
def remove_return_prefix(column):
    return column.replace('log_return_', '', 1)

LEVEL_COLUMNS = [remove_return_prefix(column) for column in ALL_RETURNS]

# Short labels improve tables and charts without changing the source columns.
LABELS = {column: remove_return_prefix(column).replace('^', '') for column in ALL_RETURNS}
FACTOR_LABELS = [LABELS[column] for column in BENCHMARKS]
TARGET_LABELS = [LABELS[column] for column in TARGETS]
FACTOR_COLORS = {
    'BTC-USD': '#F7931A', 'ETH-USD': '#627EEA',
    'DJI': '#9467BD', 'GC=F': '#D4AF37'
}

## 2. Data and reproducibility — 1 minute

The project stores source snapshots locally, so the presentation does not depend on an internet connection or a changing API response. The preprocessing pipeline:

1. loads closing prices for the selected assets;
2. aligns them to a daily calendar;
3. forward-fills non-trading days;
4. computes continuously compounded returns, $r_t = \log(P_t/P_{t-1})$;
5. retains the common sample with complete observations.

Using returns rather than price levels makes assets with very different price scales comparable and reduces the non-stationarity typically present in levels.

In [ ]:
# Ensure local snapshots reach the requested end date. Missing trailing
# observations are downloaded and merged into the existing files.
coverage_audit = ensure_yahoo_data_coverage(
    path=str(DATA_DIR), tickers=TICKERS, from_date=START_DATE, to_date=END_DATE
)

# Build the analysis dataset from the updated local market-data snapshots.
aligned_data = crypto_series_to_process(
    path=str(DATA_DIR),
    from_date=START_DATE,
    to_date=END_DATE,
    save=False,
    tickers=TICKERS
).sort_values('Date')

# Fail early with an informative error if a required source series is unavailable.
required_columns = ['Date'] + LEVEL_COLUMNS + ALL_RETURNS
missing_columns = sorted(set(required_columns) - set(aligned_data.columns))
assert not missing_columns, f'Missing required columns: {missing_columns}'

# A balanced panel ensures every model window compares the same dates across assets.
analysis_df = aligned_data[required_columns].dropna().reset_index(drop=True)
assert analysis_df[required_columns].isna().sum().sum() == 0

# This compact audit is useful to establish scope without spending presentation time
# scrolling through raw rows or printing every intermediate dataframe.
data_audit = pd.DataFrame({
    'Metric': [
        'Common sample start', 'Common sample end', 'Daily observations',
        'Target assets', 'Benchmark factors', 'Model window', 'Estimation frequency'
    ],
    'Value': [
        analysis_df['Date'].min().date(), analysis_df['Date'].max().date(),
        f'{len(analysis_df):,}', len(TARGETS), len(BENCHMARKS),
        f'{WINDOW_SIZE} days', f'Every {STEP_SIZE} days'
    ]
})

display(coverage_audit)
display(data_audit)

### Result comment: data quality and scope

- The balanced dataset contains **1,573 daily observations**.
- The common sample starts on **11 April 2020**, when Solana data becomes available; this is the binding coverage constraint.
- Removing incomplete rows makes comparisons consistent, but it also introduces **survivorship and availability bias**: conclusions apply to these selected assets and period, not to the entire crypto universe.
- Traditional markets do not trade every calendar day. Forward-filled DJI and gold prices therefore create zero weekend returns; this supports calendar alignment but should be revisited in a production pipeline.

In [ ]:
# Calculate presentation-level risk statistics. Because the aligned dataset uses
# calendar days, 365 is the consistent annualization factor for all displayed assets.
returns = analysis_df[ALL_RETURNS].rename(columns=LABELS)
annualization = 365
summary_stats = pd.DataFrame({
    'Annualized mean log return': returns.mean() * annualization,
    'Annualized volatility': returns.std() * np.sqrt(annualization),
    'Skewness': returns.skew(),
    'Excess kurtosis': returns.kurt()
})

# Measure each target's unconditional correlation with each candidate benchmark.
correlation_matrix = (
    analysis_df[TARGETS + BENCHMARKS]
    .corr()
    .loc[TARGETS, BENCHMARKS]
    .rename(index=LABELS, columns=LABELS)
)

fig, axes = plt.subplots(1, 2, figsize=(15, 5.5), gridspec_kw={'width_ratios': [1, 1.25]})

# Risk/return map: targets are orange and benchmark assets are blue.
for asset in returns.columns:
    group_color = '#E45756' if asset in TARGET_LABELS else '#4C78A8'
    axes[0].scatter(
        summary_stats.loc[asset, 'Annualized volatility'],
        summary_stats.loc[asset, 'Annualized mean log return'],
        color=group_color, s=70, alpha=0.9
    )
    axes[0].annotate(
        asset,
        (summary_stats.loc[asset, 'Annualized volatility'],
         summary_stats.loc[asset, 'Annualized mean log return']),
        xytext=(5, 4), textcoords='offset points', fontsize=8
    )
axes[0].axhline(0, color='black', linewidth=0.8)
axes[0].set(title='Risk/return profile', xlabel='Annualized volatility',
            ylabel='Annualized mean log return')
axes[0].legend(
    handles=[Line2D([0], [0], marker='o', color='w', markerfacecolor='#E45756',
                    label='Target altcoins', markersize=8),
             Line2D([0], [0], marker='o', color='w', markerfacecolor='#4C78A8',
                    label='Benchmarks', markersize=8)],
    loc='best'
)

# Annotated heatmap makes the target/factor relationship readable at a glance.
heatmap = axes[1].imshow(correlation_matrix, cmap='coolwarm', vmin=-1, vmax=1, aspect='auto')
# Set tick positions and labels separately for compatibility with older Matplotlib.
axes[1].set_xticks(range(len(FACTOR_LABELS)))
axes[1].set_xticklabels(FACTOR_LABELS, rotation=35, ha='right')
axes[1].set_yticks(range(len(TARGET_LABELS)))
axes[1].set_yticklabels(TARGET_LABELS)
axes[1].set_title('Target-to-benchmark return correlation')
for row in range(correlation_matrix.shape[0]):
    for column in range(correlation_matrix.shape[1]):
        value = correlation_matrix.iloc[row, column]
        axes[1].text(column, row, f'{value:.2f}', ha='center', va='center',
                     color='white' if abs(value) > 0.50 else 'black', fontsize=9)
fig.colorbar(heatmap, ax=axes[1], fraction=0.046, pad=0.04)

fig.suptitle('Exploratory result: high crypto risk and shared crypto-market exposure',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Keep the numerical table available for questions without making it the main slide.
display(summary_stats.round(3))

### How to read the exploratory figure

**Left panel — risk/return map**

- Each point represents one asset over the complete common sample.
- The **horizontal axis** is annualized volatility: points farther right had larger day-to-day return fluctuations and therefore higher historical risk.
- The **vertical axis** is annualized mean log return: points higher on the chart had stronger average in-sample performance.
- **Orange points** are the six target altcoins; **blue points** are the four benchmark factors.
- The upper-right region is not automatically “better”: it combines higher historical return with higher risk. The chart describes the observed sample and is not an efficient-frontier or forecasting result.

**Right panel — target-to-benchmark correlation heatmap**

- Each **row** is a target altcoin and each **column** is a benchmark factor.
- Every cell reports the Pearson correlation between the two daily return series, from **−1 to +1**.
- Redder cells indicate stronger positive co-movement, bluer cells indicate negative co-movement, and values near zero indicate a weak linear relationship.
- Correlation measures pairwise co-movement only. It does not control for the other factors and does not imply causality.

### Result comment: risk and unconditional relationships

- **Crypto-to-crypto correlation dominates:** BTC and ETH have the strongest unconditional relationships with most target returns.
- **Traditional factors are weaker unconditionally:** DJI and gold generally have smaller target-return correlations.
- **Tail risk remains important:** skewness and excess kurtosis show that averages can be influenced by rare extreme observations.

These are descriptive, in-sample statistics—not forecasts. They motivate a multivariate, time-varying attribution model.

## 3. Method: rolling Sharpe style analysis — 2 minutes

For target return $r_t$, estimate a factor-replicating portfolio:

$$r_t \approx \sum_{j=1}^{5} w_j f_{j,t} + \varepsilon_t$$

subject to:

$$w_j \ge 0 \quad\text{and}\quad \sum_{j=1}^{5} w_j = 1$$

The constrained quadratic program minimizes residual variance. The restrictions make the coefficients interpretable as a **long-only allocation across styles**, rather than unrestricted regression slopes.

- **Window:** 300 trailing daily observations.
- **Step:** seven days, providing weekly estimates and an interview-friendly run time.
- **Model quality:** $R^2 = 1 - \operatorname{Var}(\varepsilon) / \operatorname{Var}(r)$.
- **Diagnostics:** Augmented Dickey–Fuller tests assess return stationarity; approximate weight confidence intervals are calculated by the project helper.

A style weight is an attribution result, not proof of causality or a trading signal.

In [ ]:
# Confirm that return series reject the ADF unit-root null at the 5% level.
# This supports—but does not prove—the stability assumptions behind return modeling.
adf_results = create_adf_result_data_frame(analysis_df[['Date'] + ALL_RETURNS])
adf_p_values = pd.to_numeric(adf_results.loc['p-value']).rename(index=LABELS)
assert (adf_p_values < 0.05).all(), 'At least one return series fails the 5% ADF check.'

# Fit one rolling constrained model per target. `rolling_style_analysis` returns
# factor weights, approximate confidence intervals, and R-squared for each window.
style_results = {}
coefficient_names = {f'coeff_{i}': factor for i, factor in enumerate(FACTOR_LABELS)}
for target_column in TARGETS:
    target_label = LABELS[target_column]
    result = rolling_style_analysis(
        data=analysis_df,
        index_columns=BENCHMARKS,
        target_variable=target_column,
        window=WINDOW_SIZE,
        step=STEP_SIZE
    )
    style_results[target_label] = result.rename(columns=coefficient_names)

# Aggregate rolling results into one interview-friendly table. The detailed weekly
# paths remain available in `style_results` for drill-down questions.
average_weights = pd.DataFrame({
    target: result[FACTOR_LABELS].mean()
    for target, result in style_results.items()
}).T.loc[TARGET_LABELS]

fit_summary = pd.DataFrame({
    'Mean R²': {target: result['R_squared'].mean() for target, result in style_results.items()},
    'Latest R²': {target: result['R_squared'].iloc[-1] for target, result in style_results.items()},
    'Minimum R²': {target: result['R_squared'].min() for target, result in style_results.items()},
    'Maximum R²': {target: result['R_squared'].max() for target, result in style_results.items()}
}).loc[TARGET_LABELS]

model_summary = average_weights.join(fit_summary[['Mean R²', 'Latest R²']])
model_summary.index.name = 'Target'
display(model_summary.round(3))

### Result comment: model validity and fit

- Every analyzed return series rejects the ADF unit-root null at the **5% significance level**; all p-values are effectively near zero.
- Mean rolling $R^2$ ranges from **0.39 for DOGE** to **0.62 for Litecoin**. Litecoin behaves most like a replicable benchmark mix; DOGE retains the largest idiosyncratic component.
- The latest weekly estimates ending **26 July 2024** explain approximately **53% of DOGE** and **52% of ADA** variance, but only **36% of BNB** variance.
- Model fit is not constant. The min/max columns quantify how much explanatory power changes across regimes, which is why a single full-sample regression would hide important behavior.

In [ ]:
# Pair average factor composition with average explanatory power. Together, the
# panels answer both 'what drives each coin?' and 'how complete is that explanation?'
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5), gridspec_kw={'width_ratios': [1.6, 1]})

average_weights.plot(
    kind='bar', stacked=True, ax=axes[0], width=0.78,
    color=[FACTOR_COLORS[factor] for factor in FACTOR_LABELS]
)
axes[0].set(title='Average rolling style weights', xlabel='', ylabel='Weight', ylim=(0, 1))
axes[0].tick_params(axis='x', rotation=0)
axes[0].legend(title='Factor', ncol=3, loc='upper center', bbox_to_anchor=(0.5, 1.22))

ordered_fit = fit_summary['Mean R²'].sort_values()
bars = axes[1].barh(ordered_fit.index, ordered_fit.values, color='#4C78A8')
axes[1].set(title='Average model fit', xlabel='Mean rolling R²', xlim=(0, 0.70))
# `Axes.bar_label` is unavailable in older Matplotlib releases. Place labels
# explicitly so the chart works in both the Python 3.7 and modern setups.
for bar, value in zip(bars, ordered_fit.values):
    axes[1].text(value + 0.01, bar.get_y() + bar.get_height() / 2,
                 f'{value:.2f}', va='center', fontsize=9)

fig.suptitle('Main result: shared crypto beta, with meaningful unexplained risk',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### How to read the main-results figure

**Left panel — average rolling style weights**

- Each bar represents one target altcoin and summarizes all of its rolling 300-day estimates.
- Each colored segment is the target's **average weight** on one benchmark factor: BTC, ETH, DJI, or gold.
- The segments sum to **1** because the style model constrains weights to be non-negative and fully invested.
- A larger segment means that factor made a larger contribution to the minimum-variance replication mix. It is not a percentage of return earned and should not be interpreted as causality.
- Averaging makes cross-asset comparison easy, but it hides time variation; the following figure restores that time dimension.

**Right panel — average model fit**

- Each horizontal bar is the target's mean rolling $R^2$.
- $R^2$ is the fraction of target-return variance explained by the four-factor replication model.
- Longer bars indicate a target that behaved more like the selected benchmark combination—not necessarily a better investment.

### Result comment: factor composition

- **Bitcoin and Ethereum dominate the crypto exposures** across the target assets.
- **DJI and gold provide traditional-market comparisons**, although their weights are generally smaller.

A key modeling caveat is **factor correlation**. BTC and ETH are correlated, so the optimizer may redistribute weight between economically similar factors. The weights should be interpreted as a parsimonious replication mix, not as uniquely identified causal effects.

In [ ]:
# Three representative targets communicate regime behavior without overwhelming a
# 15-minute presentation: DOGE (idiosyncratic), BNB (traditional-market shift), and
# SOL (strong ETH style that later broadens toward BTC).
focus_targets = ['DOGE-USD', 'BNB-USD', 'SOL-USD']
fig, axes = plt.subplots(len(focus_targets), 1, figsize=(14, 10), sharex=True)

for axis, target in zip(axes, focus_targets):
    result = style_results[target]
    axis.stackplot(
        result.index,
        *[result[factor] for factor in FACTOR_LABELS],
        labels=FACTOR_LABELS,
        colors=[FACTOR_COLORS[factor] for factor in FACTOR_LABELS],
        alpha=0.82
    )
    axis.set_ylim(0, 1)
    axis.set_ylabel('Style weight')
    axis.set_title(target)

    # R-squared uses the same 0-1 scale but a separate axis label for clarity.
    fit_axis = axis.twinx()
    fit_axis.plot(result.index, result['R_squared'], color='black', linewidth=1.5,
                  linestyle='--', label='Rolling R²')
    fit_axis.set_ylim(0, 1)
    fit_axis.set_ylabel('R²')

legend_handles = [
    Line2D([0], [0], color=FACTOR_COLORS[factor], linewidth=7, label=factor)
    for factor in FACTOR_LABELS
]
legend_handles.append(Line2D([0], [0], color='black', linestyle='--', label='Rolling R²'))
fig.legend(handles=legend_handles, ncol=6, loc='upper center', bbox_to_anchor=(0.5, 0.98))
fig.suptitle('Rolling result: style exposures and model fit change through time',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

# Quantify the visual regime comparison for reproducible result commentary.
regime_rows = []
for target in focus_targets:
    result = style_results[target]
    midpoint = len(result) // 2
    for period, sample in [('First half', result.iloc[:midpoint]),
                           ('Second half', result.iloc[midpoint:])]:
        regime_rows.append({
            'Target': target, 'Period': period,
            'BTC weight': sample['BTC-USD'].mean(),
            'ETH weight': sample['ETH-USD'].mean(),
            'DJI weight': sample['DJI'].mean(),
            'Mean R²': sample['R_squared'].mean()
        })
regime_summary = pd.DataFrame(regime_rows).set_index(['Target', 'Period'])
display(regime_summary.round(3))

### How to read the rolling-regime figure

- The three panels focus on **DOGE, BNB, and SOL**, selected to illustrate different types of regime change without overcrowding the presentation.
- The **horizontal axis** is the ending date of each trailing 300-day estimation window. Adjacent estimates are seven days apart.
- The **left vertical axis** measures style weight. The colored areas show BTC, ETH, DJI, and gold weights; together they sum to 1 at every date.
- A widening colored band indicates that the corresponding benchmark became more important in the target's replication mix; a narrowing band indicates declining importance.
- The **black dashed line** uses the right vertical axis and shows rolling $R^2$. A rising line means the selected factors explain more of the target's recent variance; a falling line signals more unexplained, asset-specific behavior.
- Read weights and $R^2$ together: a dominant factor identifies the composition of explained behavior, while $R^2$ indicates how complete that explanation is.
- Sharp weight substitutions can partly reflect correlated benchmarks rather than a fundamental market break, so persistent changes are more informative than isolated weekly movements.

The table below the plot quantifies the visual comparison by reporting average BTC, ETH, and DJI weights and mean $R^2$ separately for the first and second halves of the rolling estimates.

The operational implication is that static labels such as “high-beta crypto” are insufficient: exposures should be monitored through time.

## 4. What this means for a client — 2 minutes

### Decision implications

1. **Do not equate more coin names with more diversification.** Much of the return variation maps back to BTC and ETH, so nominally different holdings can share the same underlying market exposure.
2. **Retain an asset-specific risk budget.** With average $R^2$ between 39% and 62%, a large share of variance remains unexplained by the selected factors.
3. **Use rolling weights as a monitoring layer.** Material shifts can trigger deeper investigation into market structure, token-specific news, liquidity, or changing investor composition.
4. **Separate attribution from prediction.** This model explains historical co-movement; it does not establish causal drivers or expected future returns.

A practical deliverable would be a dashboard that reports current factor weights, confidence bands, $R^2$, and alerts for statistically material regime changes.

## 5. Limitations and next steps — 1.5 minutes

### Current limitations

- **Calendar mismatch:** weekend values for DJI and gold are forward-filled.
- **Correlated factors:** BTC and ETH can reduce coefficient identifiability.
- **Selection bias:** six surviving large-cap altcoins are not the full investable universe.
- **Data quality:** free market-data snapshots can contain revisions, gaps, or ticker-definition changes.
- **In-sample analysis:** $R^2$ measures historical replication, not out-of-sample forecasting performance.

### Prioritized next steps

1. Re-run on matched trading dates and compare with the daily-calendar specification.
2. Add out-of-sample validation and benchmark against unconstrained and regularized regressions.
3. Test factor redundancy with condition numbers or variance-inflation diagnostics.
4. Add liquidity, momentum, volatility, macro, and on-chain factors.
5. Bootstrap rolling weights to quantify uncertainty more robustly and productionize data-quality checks.

## Conclusion — 1 minute

### Three takeaways

- **Shared exposure:** Ethereum and Bitcoin explain the largest part of the selected altcoins' replicable return behavior.
- **Meaningful differentiation:** even the best-fitting target retains substantial unexplained variance; DOGE is particularly idiosyncratic.
- **Dynamic risk:** style weights and explanatory power change materially, so portfolio conclusions should be monitored rather than treated as permanent.

**Recommended action:** use rolling style analysis as an interpretable risk-attribution tool, then validate regime changes with richer factors and out-of-sample tests before making investment decisions.

---

### Suggested 15-minute allocation

| Section | Time |
|---|---:|
| Question and data | 2 min |
| EDA and model | 4 min |
| Main and rolling results | 5 min |
| Implications, limitations, conclusion | 3 min |
| Buffer / transition to questions | 1 min |